<a href="https://colab.research.google.com/github/mathew-seby/avcs/blob/main/avcs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Google Drive to access your dataset
from google.colab import drive
drive.mount('/content/drive')

# Install required libraries for YOLO and evaluation
!pip install -q ultralytics
!pip install -q torch torchvision
!pip install -q pycocotools
!pip install -q tensorboard

# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 70.1 MB/s eta 0:00:00
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU Device: Tesla T4


In [2]:
# List files in your Google Drive to find the dataset
# This helps you see the exact filename
!ls -la "/content/drive/MyDrive/INT MSCCS 2028/Mini Project (AVCS)/Dataset/avcs_dataset.zip"

# If your file is inside a folder, check that folder
# !ls -la "/content/drive/MyDrive/your_folder_name"

-rw------- 1 root root 83753335 May 22 07:39 '/content/drive/MyDrive/INT MSCCS 2028/Mini Project (AVCS)/Dataset/avcs_dataset.zip'


In [3]:
# Import required libraries
import os
import shutil
import zipfile

# ===== UPDATE THIS WITH YOUR ACTUAL FILE ID OR PATH =====
# Since you have: https://drive.google.com/file/d/1X2Z2pB4IRZdnS3YZ6mFyK4B1IN0ztNk1/view
# First download it using gdown, OR if already in Drive, use direct path

# Option A: If file is already in your Drive (after uploading via web)
# Find it by running Cell 2 above, then update this path:
zip_file_path = '/content/drive/MyDrive/INT MSCCS 2028/Mini Project (AVCS)/Dataset/avcs_dataset.zip'  # Update filename if different

# Option B: Download directly using file ID from share link
# Uncomment these 2 lines if using direct download:
# !pip install -q gdown
# !gdown 1X2Z2pB4IRZdnS3YZ6mFyK4B1IN0ztNk1 -O /content/dataset.zip
# zip_file_path = '/content/dataset.zip'

# Extract the ZIP file to working directory
extract_path = '/content/dataset'
os.makedirs(extract_path, exist_ok=True)

print("Extracting dataset...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Find the extracted folder name (adjust if your folder has different name)
extracted_folders = os.listdir(extract_path)
print(f"Extracted folders: {extracted_folders}")

# Set final dataset path (update folder name if different)
dataset_final_path = '/content/dataset/avcs_dataset'

# Verify extraction
print(f"\n✅ Dataset extracted to: {dataset_final_path}")
print(f"Train images: {len(os.listdir(dataset_final_path + '/train/images'))}")
print(f"Train labels: {len(os.listdir(dataset_final_path + '/train/labels'))}")
print(f"Val images: {len(os.listdir(dataset_final_path + '/valid/images'))}")
print(f"Val labels: {len(os.listdir(dataset_final_path + '/valid/labels'))}")
print(f"Test images: {len(os.listdir(dataset_final_path + '/test/images'))}")
print(f"Test labels: {len(os.listdir(dataset_final_path + '/test/labels'))}")

Extracting dataset...
Extracted folders: ['avcs_dataset']

✅ Dataset extracted to: /content/dataset/avcs_dataset
Train images: 1540
Train labels: 1540
Val images: 440
Val labels: 440
Test images: 220
Test labels: 220


In [4]:
# Create YAML configuration file for YOLO training
import yaml

# Define dataset paths and classes
data_config = {
    'train': dataset_final_path + '/train/images',
    'val': dataset_final_path + '/valid/images',
    'test': dataset_final_path + '/test/images',
    'nc': 4,  # Number of classes
    'names': ['car', 'motorcycle', 'bus', 'truck']  # Class names in order
}

# Save configuration to file
config_path = '/content/data.yaml'
with open(config_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"✅ Configuration saved to: {config_path}")
print("\nConfiguration content:")
print(yaml.dump(data_config, default_flow_style=False))

✅ Configuration saved to: /content/data.yaml

Configuration content:
names:
- car
- motorcycle
- bus
- truck
nc: 4
test: /content/dataset/avcs_dataset/test/images
train: /content/dataset/avcs_dataset/train/images
val: /content/dataset/avcs_dataset/valid/images



In [5]:
# Import YOLO and time tracking
from ultralytics import YOLO
import time

# Load pre-trained YOLOv8 nano model (fastest version)
model_yolov8 = YOLO('yolov8n.pt')

# Set training parameters
epochs = 50  # Number of training iterations
img_size = 640  # Image size for training
batch_size = 16  # Images per batch

print("=" * 60)
print("TRAINING MODEL 1: YOLOv8")
print("=" * 60)

# Start timer
start_time = time.time()

# Train the model on night traffic dataset
results_yolov8 = model_yolov8.train(
    data='/content/data.yaml',
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    name='yolov8_nighttime',
    project='/content/results',
    device=0,  # Use GPU
    patience=10  # Early stopping patience
)

# Calculate training time
training_time_yolov8 = time.time() - start_time

print(f"\n✅ YOLOv8 Training completed in {training_time_yolov8/60:.2f} minutes")
print(f"Best model saved at: /content/results/yolov8_nighttime/weights/best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
TRAINING MODEL 1: YOLOv8
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, h

In [6]:
# Clone YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -qr requirements.txt
%cd ..

print("=" * 60)
print("TRAINING MODEL 2: YOLOv5")
print("=" * 60)

# Start timer
start_time = time.time()

# Run YOLOv5 training with same parameters as YOLOv8
!python yolov5/train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data /content/data.yaml \
    --weights yolov5n.pt \
    --project /content/results \
    --name yolov5_nighttime \
    --device 0 \
    --patience 10

# Calculate training time
training_time_yolov5 = time.time() - start_time

print(f"\n✅ YOLOv5 Training completed in {training_time_yolov5/60:.2f} minutes")
print(f"Best model saved at: /content/results/yolov5_nighttime/weights/best.pt")

Streaming output truncated to the last 5000 lines.
       8/49      2.34G    0.03758    0.01519   0.005982         65        640:  28% 27/97 [00:10<00:30,  2.33it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       8/49      2.34G    0.03745    0.01519   0.005926         57        640:  29% 28/97 [00:10<00:36,  1.92it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       8/49      2.34G     0.0374    0.01519   0.005972         55        640:  30% 29/97 [00:11<00:30,  2.23it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       8/49      2.34G    0.03751    

In [7]:
!git clone https://github.com/mathew-seby/avcs.git

Cloning into 'avcs'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [8]:
%cd avcs

/content/avcs


In [17]:
!git config --global user.email "mathewceeyes@gmail.com"
!git config --global user.name "mathew-seby"

In [20]:
from getpass import getpass

token = getpass('Enter GitHub token: ')
username = "mathew-seby"
repo = "avcs"

!git remote set-url origin https://{token}@github.com/{username}/{repo}.git

Enter GitHub token: ··········


In [21]:
!git push origin main

Everything up-to-date


In [22]:
!git add .
!git commit -m "Initial commit 2 models trained yolov8 & yolov5"
!git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [23]:
!pwd

/content/avcs


In [24]:
%cd /content/avcs

/content/avcs


In [25]:
!ls

README.md
